# Week 02: Data Cleaning (Used Car Listings)



## 1. Load and Look at the Data

In [26]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/raw/car_details_v3.csv")    
df = raw.copy()                                        # the copy I will clean

print("Rows and columns:", df.shape)
df.head()

Rows and columns: (8128, 13)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB


In [28]:
df.describe()

,year,selling_price,km_driven,seats
count,8128.000000,8.128000e+03,8.128000e+03,7907.000000
mean,2013.804011,6.382718e+05,6.981951e+04,5.416719
std,4.044249,8.062534e+05,5.655055e+04,0.959588
min,1983.000000,2.999900e+04,1.000000e+00,2.000000
25%,2011.000000,2.549990e+05,3.500000e+04,5.000000
50%,2015.000000,4.500000e+05,6.000000e+04,5.000000
75%,2017.000000,6.750000e+05,9.800000e+04,5.000000
max,2020.000000,1.000000e+07,2.360457e+06,14.000000


## 2. Find the Problems

I only look in this section. I change nothing yet.

### 2.1 Missing values

In [29]:
df.isnull().sum()

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
max_power        215
torque           222
seats            221
dtype: int64

About 220 values are missing in each of five columns: `mileage`, `engine`, `max_power`, `torque` and `seats`.

### 2.2 Duplicate rows

In [30]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 1202


In [31]:
# Which car names have the most extra copies?
df[df.duplicated()]["name"].value_counts().head(5)

name
BMW X4 M Sport X xDrive20d                58
Jaguar XF 2.0 Diesel Portfolio            33
Lexus ES 300h                             33
Maruti Wagon R LXI                        31
Toyota Innova 2.5 VX (Diesel) 7 Seater    31
Name: count, dtype: int64

### 2.3 Numbers stored as text

In [32]:
df[["mileage", "engine", "max_power", "torque"]].head()

,mileage,engine,max_power,torque
0,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm
1,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm
2,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)"
3,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm
4,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)"


Each value is a number followed by a unit, like `23.4 kmpl` or `1248 CC`, so pandas reads it as text.
Text cannot be averaged or compared.
`torque` is worse: it is written in many different ways (`190Nm@ 2000rpm`, `22.4 kgm at 1750-2750rpm`),
and it uses two different units. It has 441 different formats.

### 2.4 Strange values

In [33]:
print("Highest km_driven:")
df.nlargest(3, "km_driven")[["name", "year", "km_driven"]]

Highest km_driven:


,name,year,km_driven
3486,Hyundai i20 Asta 1.2,2007,2360457
1810,Mahindra XUV500 W6 2WD,2012,1500000
3508,Maruti Wagon R LXI Minor,2010,577414


In [34]:
print("Lowest km_driven:")
df.nsmallest(3, "km_driven")[["name", "year", "km_driven"]]

Lowest km_driven:


,name,year,km_driven
7913,Maruti Eeco 5 STR With AC Plus HTR CNG,2011,1
399,Honda City i-VTEC CVT V,2018,1000
2963,Maruti Swift AMT VXI,2020,1000


In [35]:
print(df["fuel"].value_counts())
print()
print(df["owner"].value_counts())

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64


- A 2007 car with 2,360,457 km, a 2012 car with 1,500,000 km, and a 2011 car with 1 km are not believable.
- `fuel` and `owner` look clean: no spelling differences.
- Five cars have the owner `Test Drive Car`. That is unusual but real, so I keep them.

## 3. Clean the Data

For every step I show what I do, then **what I decided and why**.

### Step 1: Remove duplicate rows

In [36]:
before = len(df)
df = df.drop_duplicates()
print("Rows before:", before)
print("Rows after :", len(df))

Rows before: 8128
Rows after : 6926


**Decision:** remove the duplicates and keep the first copy of each.

**Why:** the copies match in every column. If I kept them, some cars would count 30 times in every average.

### Step 2: Turn text with units into numbers

In [37]:
# "23.4 kmpl" -> split into ["23.4", "kmpl"] -> first part is the number, second part is the unit
# I take the unit first, before the text is replaced by the number
df["mileage_unit"] = df["mileage"].str.split().str[1]
df["mileage"] = pd.to_numeric(df["mileage"].str.split().str[0], errors="coerce")
df["engine"] = pd.to_numeric(df["engine"].str.split().str[0], errors="coerce")
df["max_power"] = pd.to_numeric(df["max_power"].str.split().str[0], errors="coerce")

df[["mileage", "mileage_unit", "engine", "max_power"]].head()

,mileage,mileage_unit,engine,max_power
0,23.40,kmpl,1248.0,74.00
1,21.14,kmpl,1498.0,103.52
2,17.70,kmpl,1497.0,78.00
3,23.00,kmpl,1396.0,90.00
4,16.10,kmpl,1298.0,88.20


**How it works:** `.str.split()` cuts the text at the space, `.str[0]` keeps the number part and `.str[1]` keeps the unit.
`pd.to_numeric(..., errors="coerce")` turns the number text into a real number, and turns anything broken into `NaN` (missing) instead of an error.

**Decision:** keep the numbers, and keep the mileage unit in its own column.

**Why:** mileage has two units: `kmpl` (petrol and diesel) and `km/kg` (CNG and LPG). They measure different things,
so I do not convert them. A separate `mileage_unit` column keeps this honest.

In [38]:
print(df["mileage_unit"].value_counts())

mileage_unit
kmpl     6631
km/kg      87
Name: count, dtype: int64


### Step 3: Remove the torque column

In [39]:
df = df.drop(columns=["torque"])
print("Columns now:", list(df.columns))

Columns now: ['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'mileage', 'engine', 'max_power', 'seats', 'mileage_unit']


**Decision:** drop `torque`.

**Why:** it has 441 different formats and two different units (Nm and kgm). Cleaning it needs advanced text tools,
and I do not need it for the main analysis.

**Alternative:** write a function that reads the number and converts kgm to Nm. That is possible, but too complex for this task.

### Step 4: Handle impossible values

In [40]:
print("mileage equal to 0  :", (df["mileage"] == 0).sum())
print("max_power equal to 0:", (df["max_power"] == 0).sum())

mileage equal to 0  : 15
max_power equal to 0: 3


In [41]:
# A car cannot have 0 mileage or 0 power, so treat 0 as missing
df["mileage"] = df["mileage"].replace(0, np.nan)
df["max_power"] = df["max_power"].replace(0, np.nan)

**Decision:** change a mileage or power of `0` to missing (`NaN`).

**Why:** every car has some mileage and some power, so `0` is a mistake, not a real measurement.
I cannot know the true value, so I do not guess.

In [42]:
# Keep only cars with believable kilometres (100 to 1,000,000)
before = len(df)
df = df[(df["km_driven"] >= 100) & (df["km_driven"] <= 1000000)]
print("Rows removed:", before - len(df))

Rows removed: 3


**Decision:** remove the rows with `km_driven` below 100 or above 1,000,000.

**Why:** 2.36 million km on a 2007 car would be about 180,000 km every year, which is not believable.
Only 3 rows are affected, so little data is lost.

**Alternative:** replace those values with the median. Also fine, but removing 3 rows is simpler.

### Step 5: Handle missing values

In [43]:
print("Missing values before:")
print(df.isnull().sum())

Missing values before:
name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          223
engine           208
max_power        209
seats            208
mileage_unit     208
dtype: int64


In [44]:
before = len(df)
df = df.dropna()
print("Rows before:", before)
print("Rows after :", len(df))

Rows before: 6923
Rows after : 6699


**Decision:** remove the rows that still have missing values.

**Why:** the missing values come **together**. A listing usually has either all its specifications or none of them.
Filling five columns with medians would invent a whole car. About 3% of the rows are lost, which is acceptable.

**Alternative:** fill the gaps with the median. That works for one or two missing values,
but not when the whole specification block is empty.

### Step 6: Add a brand column

In [45]:
# The brand is the first word of the name: "Maruti Swift Dzire VDI" -> "Maruti"
df["brand"] = df["name"].str.split().str[0]

# Two brands have two words
df["brand"] = df["brand"].replace({"Land": "Land Rover", "Ashok": "Ashok Leyland"})

df["brand"].value_counts().head(8)

brand
Maruti       2088
Hyundai      1206
Mahindra      706
Tata          632
Honda         361
Ford          353
Toyota        324
Chevrolet     216
Name: count, dtype: int64

**Decision:** make a new `brand` column from the first word of `name`.

**Why:** brand is useful for comparing prices, but it is hidden inside a long text.
`Land Rover` and `Ashok Leyland` would otherwise show up as `Land` and `Ashok`.

### Step 7: Tidy the column names and types

In [46]:
df = df.rename(columns={"engine": "engine_cc", "max_power": "max_power_bhp"})

# These columns hold whole numbers, so make them integers
df["engine_cc"] = df["engine_cc"].astype(int)
df["seats"] = df["seats"].astype(int)

df = df.reset_index(drop=True)
df.dtypes

name              object
year               int64
selling_price      int64
km_driven          int64
fuel              object
seller_type       object
transmission      object
owner             object
mileage          float64
engine_cc          int64
max_power_bhp    float64
seats              int64
mileage_unit      object
brand             object
dtype: object

**Decision:** put the unit in the column names and make whole-number columns integers.

**Why:** `engine_cc` tells the reader the unit. `engine` and `seats` became decimals while they had gaps; now they are whole numbers again.
`reset_index` renumbers the rows after the deleted ones.

## 4. Check the Result

Each line below stops the notebook with an error if the check fails.

In [47]:
assert df.isnull().sum().sum() == 0, "there are still missing values"
assert df.duplicated().sum() == 0, "there are still duplicates"
assert (df["mileage"] > 0).all(), "a mileage is 0 or negative"
assert (df["max_power_bhp"] > 0).all(), "a power is 0 or negative"
assert df["km_driven"].between(100, 1000000).all(), "km_driven is out of range"

print("All checks passed.")

All checks passed.


## 5. Before and After

In [48]:
print("                   raw   cleaned")
print("Rows            :", len(raw), "  ", len(df))
print("Columns         :", raw.shape[1], "    ", df.shape[1])
print("Missing values  :", raw.isnull().sum().sum(), "   ", df.isnull().sum().sum())
print("Duplicate rows  :", raw.duplicated().sum(), "   ", df.duplicated().sum())
print("Highest km      :", raw["km_driven"].max(), "", df["km_driven"].max())

                   raw   cleaned
Rows            : 8128    6699
Columns         : 13      14
Missing values  : 1100     0
Duplicate rows  : 1202     0
Highest km      : 2360457  577414


In [135]:
# The columns that were text can now be summarized
df[["selling_price", "km_driven", "mileage", "engine_cc", "max_power_bhp"]].describe().round(1)


,selling_price,km_driven,mileage,engine_cc,max_power_bhp
count,6699.0,6699.0,6699.0,6699.0,6699.0
mean,526070.6,72883.8,19.5,1430.7,87.8
std,522585.5,48642.6,3.9,493.0,31.7
min,29999.0,1000.0,9.0,624.0,32.8
25%,250000.0,38000.0,16.8,1197.0,68.0
50%,420000.0,68240.0,19.5,1248.0,81.8
75%,650000.0,100000.0,22.5,1498.0,100.0
max,10000000.0,577414.0,42.0,3604.0,400.0


## 6. Save the Cleaned Data

In [50]:
import os
os.makedirs("../data/cleaned", exist_ok=True)     # make the folder if it does not exist

df.to_csv("../data/cleaned/car_details_cleaned.csv", index=False)

check = pd.read_csv("../data/cleaned/car_details_cleaned.csv")
print("Saved. Rows and columns:", check.shape)
check.head()

Saved. Rows and columns: (6699, 14)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine_cc,max_power_bhp,seats,mileage_unit,brand
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248,74.00,5,kmpl,Maruti
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498,103.52,5,kmpl,Skoda
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497,78.00,5,kmpl,Honda
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396,90.00,5,kmpl,Hyundai
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298,88.20,5,kmpl,Maruti


## Summary

**What I did:** cleaned a file of 8,128 used-car listings.
I removed duplicates, turned text with units into numbers, dropped the messy `torque` column,
treated impossible values, removed rows with missing specifications, and added a `brand` column.

**Key decisions**
- Duplicates were removed, because the copies matched in every column.
- Impossible values were treated as missing or removed, not guessed.
- Rows with missing specifications were removed, because the specifications are missing together.
- `torque` was dropped, because it is too messy for this task.
- Expensive cars were kept, because they are real.

**Limits**
- Without an ID column, I cannot be 100% sure the duplicates are true copies.
- About 3% of the rows were lost to missing specifications.
- The `torque` information is gone.

**Output:** `data/cleaned/car_details_cleaned.csv`